# 01 — El modelo base preentrenado

**Módulo:** `src/llm_rlhf/model.py` → `PretrainedLLM`

Antes de poder *alinear* un modelo de lenguaje, necesitamos uno. Este notebook es el más corto de la serie porque la idea es corta: un LLM preentrenado es simplemente una función de texto a texto. Todo lo que sigue — SFT, modelado de recompensa, PPO/DPO/GRPO — se trata de *moldear* esa función para que sus salidas sean más útiles.

## ¿Qué hace en realidad un LLM preentrenado?

Fue entrenado con *predicción del siguiente token*: dada una ventana de texto, predecir el siguiente token. Ese es todo el objetivo de entrenamiento. Como efecto secundario de hacer esto sobre Internet, el modelo absorbe gramática, hechos y patrones de razonamiento.

Lo que **nunca se le mostró** es la diferencia entre *responder a una pregunta* y *continuar un texto que casualmente parece una pregunta*. Por eso, cuando le das una instrucción, puede:

- continuar con una lista de *más* preguntas,
- dar una respuesta tipo Wikipedia que ignora a su audiencia,
- o irse por las ramas hacia texto irrelevante.

El primer paso de alineamiento (SFT, notebook 02) le enseñará el *formato* de pregunta-respuesta. Los pasos posteriores (PPO/DPO/GRPO) le enseñarán cuáles respuestas son *mejores*.

> **Intuición clave:** un LLM no "sabe responder"; sabe *qué texto suele venir después de otro texto*. La alineación es enseñarle qué cuenta como "buena continuación" cuando lo anterior es una instrucción.

## ¿Por qué OPT-350M?

Por defecto usamos `facebook/opt-350m` porque es el modelo más pequeño que:

- aún produce inglés coherente (para que podamos *ver* diferencias cualitativas tras la alineación), y
- cabe en una GPU de portátil en fp16 (~700 MB) para que puedas correr el pipeline completo.

Cada módulo del repositorio es agnóstico al modelo — el wrapper expone la misma interfaz para cualquier LM causal de Hugging Face.

In [ ]:
from llm_rlhf import PretrainedLLM

llm = PretrainedLLM()  # facebook/opt-350m by default

## El modelo en tres líneas

`PretrainedLLM` es intencionalmente diminuto — solo hace tres cosas que necesitarás más adelante:

1. `generate(prompt)` — muestrea una continuación
2. `save_checkpoint(path)` — persiste pesos + tokenizer
3. `load_adapter(path)` — engancha un adaptador LoRA entrenado por una etapa posterior

Nada más. Cualquier cosa más interesante vive en un módulo aguas abajo.

In [ ]:
# What does an unaligned model do with an instruction?
prompt = "Explain quantum computing to a 10-year-old:"
print(llm.generate(prompt))

## Qué observar en la salida

Si corriste la celda anterior sobre un OPT-350M recién descargado, probablemente obtuviste (a) una definición tipo libro de texto de la computación cuántica, o (b) texto que se desvía hacia un *non sequitur*. Ambos son *fallos de seguimiento de instrucciones*, no fallos lingüísticos.

**Esta es la brecha que RLHF cierra.** Cada notebook posterior cierra una parte de esa brecha.

## Visualizando la predicción del siguiente token

La afirmación "un LLM es una función que asigna una distribución sobre el siguiente token" suena abstracta. Hagámosla concreta: tomemos un prompt y miremos los 10 tokens más probables que el modelo *cree* que vienen a continuación, junto con sus probabilidades.

Esto te muestra el "estado mental" del modelo en un instante: no genera un token, genera una *distribución*, y luego muestrea de ella.

In [ ]:
import matplotlib.pyplot as plt
import torch

def plot_next_token_distribution(llm, prompt: str, top_k: int = 10):
    """Show the top-k candidate next tokens and their probabilities."""
    inputs = llm.tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        logits = llm.model(**inputs).logits[0, -1]  # last position
    probs = torch.softmax(logits.float(), dim=-1)
    top_probs, top_ids = probs.topk(top_k)

    tokens = [llm.tokenizer.decode([i]) for i in top_ids.tolist()]
    # Reverse so the most-likely token sits at the top of the bar chart
    tokens = tokens[::-1]
    top_probs = top_probs.flip(0).cpu().numpy()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(range(top_k), top_probs)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels([repr(t) for t in tokens])
    ax.set_xlabel("probability")
    title = "Top-{} next tokens after: {!r}".format(top_k, prompt)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_next_token_distribution(llm, "Explain quantum computing to a 10-year-old:")

### Qué notar en el gráfico

Mira la *forma* de la distribución, no solo el token ganador:

- **Distribución plana** → el modelo está "indeciso"; muchos tokens son aproximadamente igual de plausibles. Esto es típico tras un prompt abierto.
- **Distribución concentrada** → el modelo está "seguro" de un único token. Esto ocurre dentro de frases formuladas ("Quantum computing **is**...").
- **Tokens raros** → fíjate cómo el tokenizer parte palabras y espacios; el primer token suele tener un espacio inicial (` quantum` vs `quantum`).

La *temperatura* en `generate()` reescala estas probabilidades antes de muestrear: temperatura alta aplana la distribución (más diversidad), temperatura baja la agudiza (más determinismo).

## Ejercicio: ¿qué tan "instructable" es el modelo base?

Compara cómo se comporta el modelo frente a tres tipos distintos de entrada. Antes de correr la siguiente celda, **predice** qué crees que pasará en cada caso. Luego ejecuta y compara:

1. **Una instrucción directa** — ¿la sigue, o la trata como texto a continuar?
2. **Un comienzo de historia** — aquí el modelo *debería* brillar (es exactamente para lo que fue entrenado).
3. **Una pregunta factual corta** — ¿da una respuesta, o sigue generando más preguntas?

Después, prueba a cambiar la temperatura (`GenerationConfig(temperature=...)`) y observa cómo varía la diversidad.

In [ ]:
from llm_rlhf.model import GenerationConfig

prompts = {
    "instruction": "Write a haiku about machine learning.",
    "story":       "Once upon a time, in a small village by the sea,",
    "question":    "What is the capital of France?",
}

cfg = GenerationConfig(max_new_tokens=60, temperature=0.7)
for label, p in prompts.items():
    print("--- {} ---".format(label))
    print("PROMPT:    ", p)
    print("CONTINUES: ", llm.generate(p, cfg))
    print()

## Siguiente: `02_sft.ipynb`

Haremos fine-tuning de este modelo sobre Dolly-15k para que aprenda la *forma* de un intercambio instrucción → respuesta.